# 02 — Chuẩn bị dữ liệu Bắc Kinh (G1-45, G1-46)

Notebook này **không viết lại logic**: mọi bước gọi hàm trong `src/data/beijing.py` (code dùng chung của nhóm). Mục đích là xem từng bước và kết quả.

Yêu cầu: đã chạy `python -m src.data.download_beijing` (có 12 file trong `data/raw/beijing/`).

In [1]:
import os, sys
while not os.path.exists('configs/data.yaml') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')                      # đưa thư mục làm việc về gốc repo
sys.path.insert(0, os.getcwd())
import pandas as pd
from src.utils import load_config, path_of, ROOT
from src.data.beijing import load_raw, clean
from src.data.cleaning_common import missing_table
cfg = load_config()

## Bước 1 — G1-45: gộp 12 file thành một bảng

- 4 cột `year, month, day, hour` → 1 cột `time` (giờ địa phương Bắc Kinh)
- Tên cột → snake_case (`PM2.5` → `pm25`); khí tượng gốc thêm đuôi `_native`
- Khoá duy nhất: `(city, station_id, time)`; hàm tự kiểm tra không trùng, không thiếu giờ

In [2]:
raw = load_raw(ROOT / cfg['paths']['raw_beijing'])
print(raw.shape)
raw.head()

(420768, 15)


,city,station_id,time,pm25,pm10,so2,no2,co,o3,temp_native,pres_native,dewp_native,rain_native,wd_native,wspm_native
0,beijing,Aotizhongxin,2013-03-01 00:00:00,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4
1,beijing,Aotizhongxin,2013-03-01 01:00:00,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7
2,beijing,Aotizhongxin,2013-03-01 02:00:00,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6
3,beijing,Aotizhongxin,2013-03-01 03:00:00,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1
4,beijing,Aotizhongxin,2013-03-01 04:00:00,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0


In [3]:
# Mỗi trạm phải có đúng 35.064 giờ, từ 01/03/2013 00:00 đến 28/02/2017 23:00
raw.groupby('station_id')['time'].agg(['min', 'max', 'count'])

,min,max,count
station_id,,,
Aotizhongxin,2013-03-01,2017-02-28 23:00:00,35064
Changping,2013-03-01,2017-02-28 23:00:00,35064
Dingling,2013-03-01,2017-02-28 23:00:00,35064
Dongsi,2013-03-01,2017-02-28 23:00:00,35064
Guanyuan,2013-03-01,2017-02-28 23:00:00,35064
Gucheng,2013-03-01,2017-02-28 23:00:00,35064
Huairou,2013-03-01,2017-02-28 23:00:00,35064
Nongzhanguan,2013-03-01,2017-02-28 23:00:00,35064
Shunyi,2013-03-01,2017-02-28 23:00:00,35064


## Bước 2 — Thiếu dữ liệu TRƯỚC khi làm sạch (% giờ thiếu theo trạm)

In [4]:
cols = ['pm25', 'temp_native', 'pres_native', 'dewp_native', 'rain_native', 'wspm_native']
missing_table(raw, cols)

,pm25,temp_native,pres_native,dewp_native,rain_native,wspm_native
station_id,,,,,,
Aotizhongxin,2.64,0.06,0.06,0.06,0.06,0.04
Changping,2.21,0.15,0.14,0.15,0.15,0.12
Dingling,2.22,0.15,0.14,0.15,0.15,0.12
Dongsi,2.14,0.06,0.06,0.06,0.06,0.04
Guanyuan,1.76,0.06,0.06,0.06,0.06,0.04
Gucheng,1.84,0.15,0.14,0.15,0.12,0.12
Huairou,2.72,0.15,0.15,0.15,0.16,0.14
Nongzhanguan,1.79,0.06,0.06,0.06,0.06,0.04
Shunyi,2.60,0.15,0.15,0.15,0.15,0.13


## Bước 3 — G1-46: làm sạch theo CÙNG quy tắc với HealthyAir

1. PM2.5 ≤ 0 → thiếu
2. **Không cắt** outlier PM2.5 (giống TV1): giá trị cao là sự kiện thật cần dự báo
3. Nội suy tuyến tính khoảng trống **≤ 3 giờ** trong từng trạm, đánh dấu `is_imputed = 1`; khoảng trống dài hơn giữ nguyên là thiếu

In [5]:
bj, log = clean(raw, cfg)
log

{'n_rows': 420768,
 'pm25_nonpositive_to_nan': 0,
 'pm25_train_q999_tham_khao': 564.0,
 'pm25_gio_vuot_q999': 411,
 'pm25_clipped': False,
 'pm25_imputed_rows': 3454}

In [6]:
# Ví dụ một đoạn được nội suy: giá trị nối thẳng giữa 2 đầu, cờ is_imputed = 1
i = bj.index[bj['is_imputed'] == 1][0]
bj.loc[i - 2:i + 3, ['station_id', 'time', 'pm25', 'is_imputed']]

,station_id,time,pm25,is_imputed
1507,Aotizhongxin,2013-05-02 19:00:00,87.0,0
1508,Aotizhongxin,2013-05-02 20:00:00,82.0,0
1509,Aotizhongxin,2013-05-02 21:00:00,124.0,1
1510,Aotizhongxin,2013-05-02 22:00:00,166.0,0
1511,Aotizhongxin,2013-05-02 23:00:00,90.0,0
1512,Aotizhongxin,2013-05-03 00:00:00,69.0,0


## Bước 4 — So sánh % thiếu PM2.5 trước / sau

In [7]:
cmp = pd.DataFrame({'truoc_%': missing_table(raw, ['pm25'])['pm25'],
                    'sau_%': missing_table(bj, ['pm25'])['pm25']})
cmp

,truoc_%,sau_%
station_id,,
Aotizhongxin,2.64,1.87
Changping,2.21,1.46
Dingling,2.22,1.51
Dongsi,2.14,1.36
Guanyuan,1.76,1.04
Gucheng,1.84,1.12
Huairou,2.72,1.85
Nongzhanguan,1.79,0.85
Shunyi,2.60,1.68


## Bước 5 — Lưu kết quả

Hai file `.parquet` nằm trong `data/interim/` (không commit, ai cần thì chạy lại). Cách nhanh hơn để làm cả 2 bước: `python -m src.data.beijing`.

In [8]:
raw.to_parquet(path_of(cfg, 'beijing_raw_parquet'), index=False)
bj.to_parquet(path_of(cfg, 'beijing_clean_parquet'), index=False)
print('Đã lưu beijing_raw.parquet và beijing_clean.parquet')

Đã lưu beijing_raw.parquet và beijing_clean.parquet
